In [1]:
%cd ..

/Users/niccolotogni/Repo/prompt-forge


In [2]:
import logging
import os

from openai import AzureOpenAI
from dotenv import load_dotenv

from prompt_forge import (
    Project,
    LLMMessage,
    LLMResponse,
)

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)


# ── Azure OpenAI client ───────────────────────────────────────────────────────
class AzureClient:
    ENDPOINT = "https://sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com/"
    API_VERSION = "2024-12-01-preview"
    DEPLOYMENT = "gpt-5.3-chat"  # ← change to your deployment name

    def __init__(self):
        self.client = AzureOpenAI(
            api_version=self.API_VERSION,
            azure_endpoint=self.ENDPOINT,
            api_key=os.environ["AZURE_OPENAI_API_KEY"],
        )

    def complete(self, messages: list[LLMMessage], **kwargs) -> LLMResponse:
        # Strip kwargs that AzureOpenAI doesn't accept (e.g. custom keys)
        allowed = {"temperature", "max_tokens", "top_p", "frequency_penalty", "presence_penalty"}
        oai_kwargs = {k: v for k, v in kwargs.items() if k in allowed}

        resp = self.client.chat.completions.create(
            model=self.DEPLOYMENT,
            messages=[{"role": m.role, "content": m.content} for m in messages],
            **oai_kwargs,
        )
        return LLMResponse(
            text=resp.choices[0].message.content,
            usage={
                "input_tokens": resp.usage.prompt_tokens,
                "output_tokens": resp.usage.completion_tokens,
            },
        )


In [3]:
# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR = "examples/extraction_example"        # Directory containing example subfolders (001/, 002/, ...)
# PROJECT_DIR = ".projects/extraction_example"  # Directory to store project files (prompt versions, logs, etc.)

BUNDLE_SCHEMA = {
    "input": ".txt",
    "expected_output": ".json",
}

SEED_PROMPT = (
    "You are a helpful assistant. Process the input and produce the expected output."
)

CONTEXT = (
    "Debug task: echo or transform text inputs."
)

# Set to "none" to skip evaluation (faster iteration), or use
# "similarity", "exact_match", "json_fields", "llm_judge"
EVAL_STRATEGY = "json_fields"


In [4]:
from prompt_forge.storage import SQLiteStore

sqlite_store = SQLiteStore(".projects/extraction_example_sqlite/store.db")

In [5]:
load_dotenv()
llm = AzureClient()
project = Project(
    name="debug_project",
    llm=llm,
    store=sqlite_store,
)

project.set_bundle_schema(**BUNDLE_SCHEMA)
project.set_context(CONTEXT)
project.set_seed_prompt(SEED_PROMPT)

n = project.add_examples_from_directory(DATA_DIR)
print(f"\nLoaded {n} examples")

if n == 0:
    print(
        "\nNo examples found. Create them under:\n"
        f"  {DATA_DIR}/001/input.txt\n"
        f"  {DATA_DIR}/001/expected_output.txt\n"
        "Then re-run."
    )


2026-03-13 10:56:16,835 [INFO] prompt_forge.project: Bundle schema set: {'input': '.txt', 'expected_output': '.json'}
2026-03-13 10:56:16,838 [INFO] prompt_forge.project: Seed prompt set.
2026-03-13 10:56:16,839 [INFO] prompt_forge.bundle: Loading bundles from subdirectories in examples/extraction_example...
2026-03-13 10:56:16,843 [INFO] prompt_forge.project: Loaded 20 example bundles from examples/extraction_example



Loaded 20 examples


In [6]:
from prompt_forge.training.pipeline import TrainingConfig
from prompt_forge.utils import train_val_split

SEED = 42

train_bundles, val_bundles = train_val_split(project.bundles, val_ratio=0.1, seed=SEED)
train_config = TrainingConfig(
    batch_size=20,
    max_iterations=1,
    inference_temperature=1,
    val_max_tokens=5000,
    seed=SEED
)
print(f"\nStarting training on {project.num_examples} examples...")

report = project.train(
    config=train_config,
    eval_strategy=EVAL_STRATEGY,
    train_bundles=train_bundles,
    val_bundles=val_bundles
)

print("\n── Training Report ─────────────────────────────────────────")
print(f"Final version : v{report.final_version}")
print(f"Final score   : {report.final_score}")
print(f"Total tokens  : {report.total_tokens_used:,}")
print(f"Refinement rec: {report.refinement_recommended}")
print()


2026-03-13 10:56:40,487 [INFO] prompt_forge.training.pipeline: Starting training: 18 train examples, val_size=2, batch_size=20, max_iterations=1
2026-03-13 10:56:40,488 [INFO] prompt_forge.training.pipeline: 
Iteration 1/1
2026-03-13 10:56:40,488 [INFO] prompt_forge.training.pipeline: Selected batch: ['spec_03', 'spec_09', 'spec_13', 'spec_20', 'spec_05', 'spec_17', 'spec_16', 'spec_08', 'spec_15', 'spec_19', 'spec_11', 'spec_06', 'spec_18', 'spec_02', 'spec_07', 'spec_12', 'spec_14', 'spec_10']
2026-03-13 10:56:40,490 [INFO] prompt_forge.inference.agent: Batch inference: 2 inputs → 1 chunk(s)
2026-03-13 10:56:40,611 [DEBUG] httpcore.connection: connect_tcp.started host='sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com' port=443 local_address=None timeout=5.0 socket_options=None



Starting training on 20 examples...


2026-03-13 10:56:40,782 [DEBUG] httpcore.connection: connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x10f5b30e0>
2026-03-13 10:56:40,782 [DEBUG] httpcore.connection: start_tls.started ssl_context=<ssl.SSLContext object at 0x10ef8c3b0> server_hostname='sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com' timeout=5.0
2026-03-13 10:56:41,027 [DEBUG] httpcore.connection: start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x10f50cb90>
2026-03-13 10:56:41,028 [DEBUG] httpcore.http11: send_request_headers.started request=<Request [b'POST']>
2026-03-13 10:56:41,029 [DEBUG] httpcore.http11: send_request_headers.complete
2026-03-13 10:56:41,030 [DEBUG] httpcore.http11: send_request_body.started request=<Request [b'POST']>
2026-03-13 10:56:41,030 [DEBUG] httpcore.http11: send_request_body.complete
2026-03-13 10:56:41,031 [DEBUG] httpcore.http11: receive_response_headers.started request=<Request [b'POST']>
2026-03-13 10:56:57,299 [DEBUG] h


── Training Report ─────────────────────────────────────────
Final version : v1
Final score   : 0.96
Total tokens  : 18,473
Refinement rec: False



In [7]:
for r in report:
    status = "✓" if r.improved else "✗"
    score_str = (
        f"{r.score_before:.3f} → {r.score_after:.3f}"
        if r.score_before is not None and r.score_after is not None
        else "no eval"
    )
    tokens_str = f"  [{r.tokens_used:,} tok]" if r.tokens_used else ""
    print(f"  Iter {r.iteration}: {score_str} {status}{tokens_str}")
    print(f"    Learned: {r.learnings[:120]}")

print("\n── Prompt versions ─────────────────────────────────────────")
for v in project.list_versions():
    score = f"score={v.eval_score:.3f}" if v.eval_score is not None else "no score"
    print(f"  v{v.version}  {score}  {(v.training_log_entry or '')[:80]}")

print("\n── Inference test ──────────────────────────────────────────")
agent = project.get_inference_agent()
print(agent.prompt_info)
result = agent.run(input_text="Hello, this is a test input.")
print(f"Output: {result}")


2026-03-13 10:59:28,948 [INFO] prompt_forge.inference.agent: Loaded prompt version 1 (score: 0.96)
2026-03-13 10:59:28,949 [DEBUG] httpcore.connection: close.started
2026-03-13 10:59:28,950 [DEBUG] httpcore.connection: close.complete
2026-03-13 10:59:28,950 [DEBUG] httpcore.connection: connect_tcp.started host='sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com' port=443 local_address=None timeout=5.0 socket_options=None


  Iter 1: 0.000 → 0.960 ✓  [17,242 tok]
    Learned: - Many specifications require unit normalization including inches, feet, meters, pounds, tons, and Fahrenheit; consisten

── Prompt versions ─────────────────────────────────────────
  v0  no score  Seed prompt
  v1  score=0.960  - Many specifications require unit normalization including inches, feet, meters,

── Inference test ──────────────────────────────────────────
Prompt: 223 lines, 6177 chars, output_schema=['product_name', 'manufacturer', 'part_number', 'year_of_manufacture', 'machine_type', 'application_sector', 'power_supply', 'dimensions', 'weight_kg', 'operating_conditions', 'performance', 'safety_certifications', 'communication_interfaces', 'maintenance_interval_hours', 'warranty_years']


2026-03-13 10:59:29,419 [DEBUG] httpcore.connection: connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x10f50d6d0>
2026-03-13 10:59:29,421 [DEBUG] httpcore.connection: start_tls.started ssl_context=<ssl.SSLContext object at 0x10ef8c3b0> server_hostname='sinte-mhrqdmfc-eastus2.cognitiveservices.azure.com' timeout=5.0
2026-03-13 10:59:29,822 [DEBUG] httpcore.connection: start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x10f6fc3e0>
2026-03-13 10:59:29,823 [DEBUG] httpcore.http11: send_request_headers.started request=<Request [b'POST']>
2026-03-13 10:59:29,824 [DEBUG] httpcore.http11: send_request_headers.complete
2026-03-13 10:59:29,824 [DEBUG] httpcore.http11: send_request_body.started request=<Request [b'POST']>
2026-03-13 10:59:29,825 [DEBUG] httpcore.http11: send_request_body.complete
2026-03-13 10:59:29,825 [DEBUG] httpcore.http11: receive_response_headers.started request=<Request [b'POST']>
2026-03-13 10:59:36,594 [DEBUG] h

Output: {'product_name': None, 'manufacturer': None, 'part_number': None, 'year_of_manufacture': None, 'machine_type': None, 'application_sector': None, 'power_supply': {'voltage_v': None, 'phases': None, 'frequency_hz': None, 'power_rating_kw': None}, 'dimensions': {'length_mm': None, 'width_mm': None, 'height_mm': None}, 'weight_kg': None, 'operating_conditions': {'temperature_min_c': None, 'temperature_max_c': None, 'humidity_max_pct': None}, 'performance': {'max_speed_rpm': None, 'throughput': None, 'accuracy_mm': None, 'noise_level_db': None}, 'safety_certifications': [], 'communication_interfaces': [], 'maintenance_interval_hours': None, 'warranty_years': None}


In [8]:
print(project.list_versions()[2].prompt_text)

IndexError: list index out of range

In [11]:
print(report.iterations[0].learnings)

- Some brands appearing in all caps should be normalized to stylized casing (e.g., LOGIMOVE → LogiMove, PYROMAX → PyroMax, CHILLPRO → ChillPro).
- Safety certifications often include descriptors like “Listed” or “certified” that must be removed while keeping the core standard.
- Battery-powered machines should still use mains charging voltage for the supply field but may use drive/motor power as the power rating.
- Throughput must return null when the text explicitly states that production rate varies with conditions.


In [12]:
print(report.iterations[0].issues)

- Brand capitalization normalization still relies on heuristic interpretation; a definitive brand dictionary would improve consistency.
- Some borderline cases remain for dimension derivation (working envelope vs physical size) when both appear but neither is clearly labeled as overall dimensions.
